# Volatility Regime ML - Research Notebook

**Source**: QC Strategy Library - Volatility Harvest ML

**Concept**: VIX regime detection with RandomForest ML overlay for tactical allocation.
Uses VIX statistics (z-score, percentile, SMA ratios) combined with SPY momentum and volatility features
to classify market regimes and adjust allocations dynamically.

In [ ]:
from AlgorithmImports import *
qb = QuantBook()

# Define universe
tickers = ["SPY", "TLT", "GLD", "BIL"]
for t in tickers:
    qb.AddEquity(t, Resolution.Daily)

print(f"Universe: {tickers}")
print(f"Signal: VIX via CBOE custom data")

## Strategy Logic

1. **Feature Engineering**: 11 features from VIX stats + SPY momentum/volatility
2. **ML Model**: RandomForestClassifier trained monthly on 2+ years of data
3. **Regime Detection**: 5 allocation states based on VIX level and SPY price action
4. **ML Overlay**: Bullish probability > 0.6 tilts allocations toward equities
5. **Daily signal check**: 30 minutes after market open

In [ ]:
import numpy as np

# Fetch historical data
spy_hist = qb.History("SPY", timedelta(252 * 12), Resolution.Daily)
print(f"SPY data shape: {spy_hist.shape}")

# Calculate VIX-style features on SPY volatility
close = spy_hist["close"]
returns = close.pct_change().dropna()
realized_vol = returns.rolling(21).std() * np.sqrt(252)

print(f"\nRealized volatility stats (annualized):")
print(f"  Mean: {realized_vol.mean():.2%}")
print(f"  Median: {realized_vol.median():.2%}")
print(f"  Max: {realized_vol.max():.2%}")
print(f"  Current: {realized_vol.iloc[-1]:.2%}")

In [ ]:
# Analyze regime transitions
vol_percentile_80 = realized_vol.expanding().apply(lambda x: np.percentile(x, 80)).iloc[-1]
vol_sma = realized_vol.rolling(20).mean()

# Classify regimes
regime = pd.Series(index=realized_vol.index, dtype=str)
regime[realized_vol > vol_percentile_80] = "High Vol"
regime[(realized_vol > vol_sma * 1.2) & (regime != "High Vol")] = "Rising Vol"
regime[realized_vol < realized_vol.rolling(50).mean() * 0.7] = "Low Vol"
regime[regime.isna()] = "Normal"

print("Regime distribution:")
print(regime.value_counts())
print(f"\nCurrent regime: {regime.iloc[-1]}")

## ML Feature Analysis

The strategy uses 11 features for the RandomForest classifier:
- **VIX features** (5): level, z-score, percentile, short/medium-term ratios
- **SPY momentum** (4): 5d, 10d, 20d returns, price vs SMAs
- **SPY volatility** (1): annualized 21-day vol
- **Combined** (1): cross-asset signal

The model is retrained monthly to adapt to changing market dynamics.

In [ ]:
# Simulate feature correlation with forward returns
forward_21d = close.pct_change(21).shift(-21)

features = pd.DataFrame({
    "spy_5d_ret": close.pct_change(5),
    "spy_10d_ret": close.pct_change(10),
    "spy_20d_ret": close.pct_change(20),
    "spy_vs_sma50": close / close.rolling(50).mean(),
    "spy_vs_sma200": close / close.rolling(200).mean(),
    "realized_vol": realized_vol,
})

# Correlation with forward returns
corr = features.join(forward_21d.rename("forward_21d")).corr()["forward_21d"].drop("forward_21d")
print("Feature correlation with 21-day forward returns:")
for feat, c in corr.sort_values(ascending=False).items():
    print(f"  {feat}: {c:.3f}")

## Key Observations

- **VIX as regime signal** is well-documented in academic literature
- **ML overlay** adds a data-driven tilt to rule-based regime allocation
- **Monthly retraining** prevents model staleness while avoiding overfitting
- **Multiple allocation states** provide granular risk management

### Strengths
- Combines VIX regime detection with ML-based bullish/bearish signal
- Diversified across equities, bonds, gold, and cash
- Monthly retraining adapts to structural breaks

### Risks
- VIX data quality depends on CBOE custom data source availability
- RandomForest may overfit on limited training data (2-year minimum)
- Regime detection lag: VIX is backward-looking
- Complex interaction between 5 regime rules and ML overlay

In [ ]:
# Allocation breakdown by regime
print("=" * 60)
print("REGIME ALLOCATION TABLE")
print("=" * 60)
print(f"{'Regime':<25} {'SPY':>6} {'TLT':>6} {'GLD':>6} {'BIL':>6}")
print("-" * 60)
print(f"{'VIX spike + oversold':<25} {'100%':>6} {'0%':>6} {'0%':>6} {'0%':>6}")
print(f"{'  (with ML bullish)':<25} {'85%':>6} {'0%':>6} {'15%':>6} {'0%':>6}")
print(f"{'VIX low + extended':<25} {'40%':>6} {'30%':>6} {'20%':>6} {'10%':>6}")
print(f"{'VIX elevated + falling':<25} {'70-85%':>6} {'10%':>6} {'5-20%':>6} {'0%':>6}")
print(f"{'VIX rising':<25} {'30%':>6} {'40%':>6} {'20%':>6} {'10%':>6}")
print(f"{'Trend (above 200 SMA)':<25} {'60-70%':>6} {'25%':>6} {'5-15%':>6} {'0%':>6}")
print(f"{'Trend (below 200 SMA)':<25} {'30%':>6} {'40%':>6} {'20%':>6} {'10%':>6}")

In [ ]:
# Backtest results placeholder
print("=" * 50)
print("BACKTEST RESULTS (11-year: 2008-2025)")
print("=" * 50)
print("Run backtest via QC MCP to populate metrics:")
print("  create_compile -> create_backtest -> read_backtest")
print()
print("Expected characteristics:")
print("  - CAGR: ~10-15% (VIX regime timing)")
print("  - Max Drawdown: ~15-20%")
print("  - Sharpe: ~0.8-1.2")
print("  - Rebalance frequency: Daily signal check")
print("  - ML retraining: Monthly")
print("  - Universe: 4 ETFs + VIX")